# Shape loss visualization

## Pareto front

In [ ]:
import pandas as pd
from paretoset import paretoset
import numpy as np
import matplotlib.pyplot as plt

metrics = pd.read_csv(
    "../../benchmarks/v1/shape_features/minirocket.sigmoid.csv",
)
shape_loss_5p = np.load("../../benchmarks/v1/shape_loss/minirocket.sigmoid.5p_idx.npy")
local_shape_loss_5p = np.load(
    "../../benchmarks/v1/local_shape_loss/minirocket.sigmoid.5p_idx.npy"
)
rest = np.setdiff1d(
    np.arange(len(metrics)), np.union1d(shape_loss_5p, local_shape_loss_5p)
)

pareto_metrics = metrics[paretoset(metrics[["H", "b"]])]
pareto_sorted = pareto_metrics.iloc[np.argsort(pareto_metrics["H"])]
pareto_sorted = pareto_sorted[["H", "b"]].to_numpy()

plt.scatter(
    metrics["H"].iloc[rest],
    metrics["b"].iloc[rest],
    label="Shape metrics",
    color="tab:blue",
)
plt.scatter(
    metrics["H"].iloc[local_shape_loss_5p],
    metrics["b"].iloc[local_shape_loss_5p],
    label=r"$\leq \mathcal{L}_\text{local}^\text{10\%}$",
    color="tab:orange",
)
plt.scatter(
    metrics["H"].iloc[shape_loss_5p],
    metrics["b"].iloc[shape_loss_5p],
    label=r"$\leq \mathcal{L}_\text{shape}^\text{10\%}$",
    color="tab:green",
)
plt.fill(
    np.concatenate(
        [pareto_sorted[:1, 0], pareto_sorted[:, 0], [plt.xlim()[1], plt.xlim()[1]]]
    ),
    np.concatenate(
        [[plt.ylim()[1]], pareto_sorted[:, 1], [pareto_sorted[-1, 1], plt.ylim()[1]]]
    ),
    alpha=0.3,
    label="Pareto front",
    color="tab:red",
)

xmin, xmax = plt.xlim()
x_range = xmax - xmin
plt.xlim(xmin, 1.2)

ymin, ymax = plt.ylim()
y_range = ymax - ymin
plt.ylim(ymin, ymax - 0.05 * y_range)

plt.xlabel(r"$H_\text{app}$")
plt.ylabel(r"$b_\text{edge}$")
plt.legend(loc="upper right")
plt.show()

## Profiles with small loss

In [ ]:
from heavyedge import ProfileData

with ProfileData(
    "../../benchmarks/v1/local_shape_loss/minirocket.sigmoid.5p_profiles.h5"
) as data:
    Ys, _, _ = data[:]
    Ys /= np.sum(Ys, axis=1, keepdims=True)
    lines = plt.plot(np.arange(Ys.shape[1]), Ys.T, color="tab:orange")
    lines[0].set_label(r"$\leq \mathcal{L}_\text{local}^\text{10\%}$")

with ProfileData(
    "../../benchmarks/v1/shape_loss/minirocket.sigmoid.5p_profiles.h5"
) as data:
    Ys, _, _ = data[:]
    Ys /= np.sum(Ys, axis=1, keepdims=True)
    lines = plt.plot(np.arange(Ys.shape[1]), Ys.T, color="tab:green")
    lines[0].set_label(r"$\leq \mathcal{L}_\text{shape}^\text{10\%}$")

plt.legend()
plt.show()